# Joint Shortcut / Cross-Domain Audit + Federated Histogram Boosting

**One self-contained Kaggle notebook.** It runs two linked experiments on Edge-IIoTset:

**Experiment A — shortcut and generalization audit:** four leave-one-attack-family-out folds; five prespecified feature groups; immutable validation/test membership per seed and fold; recalibrated only with source validation; optional external-domain evaluation **only** if the user attaches an independent dataset with compatible column names.

**Experiment B — genuine statistics-sharing FL tree training:** a federated logistic gradient-boosted decision-stump model aggregates *per-client gradient and Hessian histograms*, never raw records; compared with centralized PooledLightGBM and independent LocalLightGBM ensemble. It is **not** full federated LightGBM, and has **no cryptographic secure aggregation or differential privacy guarantee**.

Key caveats: the five clients are simulated on one source dataset; one-family holdout is not cross-network generalization; random sampling creates artificial test prevalence, so precision/AUPRC are not deployment estimates. External data must be genuinely independent, have equivalent feature extraction, and matched target semantics. The workbook records failures rather than inventing missing results.

All code and source comments are in English.

In [ ]:
from pathlib import Path
import os, sys, base64, zlib, subprocess, json, shutil

WORK = Path('/kaggle/working') if Path('/kaggle/working').is_dir() else Path('/mnt/data/edge_joint_notebook_smoke')
WORK.mkdir(parents=True, exist_ok=True)

# Leave this empty unless an independently sourced, truly compatible CSV is mounted.
# Example: '/kaggle/input/my-independent-network-dataset/flows.csv'
EXTERNAL_DATASET_CSV = os.getenv('EDGE_EXTERNAL_CSV', '')

ENV = os.environ.copy()
ENV['EDGE_AUDIT_ROOT'] = str(WORK)
ENV['EDGE_EXTERNAL_CSV'] = EXTERNAL_DATASET_CSV
for key in ('OPENBLAS_NUM_THREADS','OMP_NUM_THREADS','MKL_NUM_THREADS','NUMEXPR_NUM_THREADS'):
    ENV[key] = '1'

print('Working directory:', WORK)
print('External dataset:', EXTERNAL_DATASET_CSV if EXTERNAL_DATASET_CSV else 'Not attached (explicitly reported)')

In [ ]:
import base64, zlib
base_source = zlib.decompress(base64.b85decode('c-qx{dvn`HvhV-;6i8CL0Bi^%WjpI7gRP_3a-xb~S911bFE0xP5g>^O1P}m}$oP0a`|V%%3@{)m#on{$R^3#^0tfT#>FMb=zW(~_^LUXQUtAO)>ZGye=_ZMfo1~`4Bx=@GVnumu#Ywy_vvd|UX^~skNj_VyqUy#!x=iXMs%A?oPNPL$)J-}YS#`QvXHk>H)~nZ^wWx}9NzXNP(A1tai%R|(t*9VhcvfCmvn)ziwG~|#>xKp^vtqkS^2REw<TlD^xWqp?&lg!*FOQ2n+j>^r(7d9o$P>@{DOxPD#40K)D`rvVTW<?%nZ|LFTUg047S^nD8q7cX`s=Tc(p6bhjaAe~b7~XbBwH)ZEysTM{`~Ftug|W+w?DrLKm2%ke)jUp_Ut!5z1}IIU+2HPzuYN(_vU@C#2p>c7=Du9rd5$oFwQivhqfMn9<8%xNz2HR>d5>-O`_{8aiS~>n`M<maUFc9)`_cI*4x@^(pBR9S{J$Zds@!ZEb*EunkCoK?8d98pJL@j)uN25I`OK+TSoOVORv4f%$wD>UfLv8QxqA2pej~Y88vuoss7*7PkAvbvMiZl*K7UqMX}DQwVq49UX@!bs;#`#MJ2YTMp+r_vZf#`kv{oR_m<~Ip`Yive7??k*eF9kucT6alM%M%KA}Zf6|<zSX-{>F%i?-nH&=uJ0H;}<Dy*AC*Sn&>S#rT;I~7++Q>C+Zr8Gnst!H69D+q$oZBj*xB&5M-X-!j8<)}4VSGUPdUqzf`VVx-O={|qJX}DUh=kqMN!ihLKx_tNULjaIFge{AtA%t=4^OrxIhi5;(y!a5}J)zgh4Ws$VT~Xc8vTZA!^F}#)Z(KjE!#J%R*GjTFvFwvo-kiiy6P+Y<*u#|eZ&9VqHsteSyYA7upFac`=fnm%05u3N+FEqT2{~(fY*96Tb%U-6vFLybUV6zUt(&m8QCRxo?8T4gs=J*5XVGk#gtyQ9@+Py7UcCAth|^i))JZ}(J{fpVPrWBk2Y;D*^JtZ3X+nkedxBd?`<mkf6Chr`EUv<zU%scutN-<yI{#HbOusLqye?LEG&g(d9a(ueqdqXrXS7Z0JjO%aBxM~u_pYO6whZg^_aqpeKKHUJ82E$1@R=v1BRGBPC9CTsj%ie?LG|kd?T@Tgw4uF^@~|rI>fkMCP_&NICS;lh3@NUoYnpzur9Ow>db8y^zmcJy4+aCTq?1RWD8n+TLSE@Mc!n25S<^+1SHbWp3XUugfO=VEaoEs#P8xv89}GSE=h@Vwtw?9fENQ|hE0+-shY2duP1uL(b{)-=Cj$acvcb_z;*bt)LJbK>OxeS-AmT32&S^PRb-pIzQ`2O)Cn}Imj3NM-ksPgFCGpwq;>~Mpiah^x_Li#PZb)yF-~qJqiGJBc+UC|_^uK!Z?x%Cx4L~}Hq7u&d>ynT;X|d)(Z0V!O)a!8KiN=Yc!iiH_k@4R>yJT+CU+CwTV8hTKJhamDb9HUdB5B*OJbM~U==eAr*TQAB!A)gcD*Mf+K3}vBE!~|S9mUDqiit!!rMGcMOfO2;zoQLE!Y0`?4z3U%HEL($rGy=MLx7-WxR8P@T3yGHbvv?deQL5Il0wrb@|WWRo+Zw$mzT7b9O$ITP2E(^t?PPpZ5Moa>~%ALPFsw*U{Q3hIW?9k_mqz^Di~8wJ0)@}tfG7!ftH$ugxf%4eL*^_G<Qxt0?n*khcnz!G*^PyDN#4fSz683aRllZ-9~A~m#jN7-WVf=GE1IxUQ{dEfl-wZ=I54e`@a@6+>EzmhAqLGH?E%*cS$8Mn$a*(Ml*D10#|3R&Ivc8s`QJJpn?O1AIm!*RDph_vsFoJ`Yp`bU+a8bCo%n6Hcjcz3OYD>L+|rNv(%+cR)@G(=yx+KeY*VB6IIUz=lyw_?-y2evtL+m^Mi$5xAQDo)aJ7>B+ZPT)-lbBc$1JQLlaR!oY($u)z<%QZTx7qWukr(P-)leRa)0V#CwHJSS7!$6Q1qi7a=ci^;^?m)|iw(qboQkYF+QVXn!x`s`p!_&X22lp(e*a>GU>5XuNNVGM$+P#1`Q3x&whc6T?Y!i|y&rH`aTi;U%#R^OQK>WHZawM0c%=mnVdSNd`0}LQe3uqNX8$T41h|in#G4i|dgU-4<zVt(n`ATYxy@VnYPaKl=XcB`}NI(b(J}{Uwf?mrpEvvM8!8D0np^N=slQE**bULbz%s4>1E+O2TEKe^IgiRTt}OmPiK#^)zQ$fPZ7Vi5#r(Jg=fv!U)j*pfhmIFq+fx@Y7g5@xz+`T}3lB6+FpYl4hyO@FZo7^5AttFf+P9;epRr*1Ae}%Z(zn7JjC~RfWXR<xO|Qikmbsur!ZWNpIf`B-tH>c*kWChgEdfy{Sv$n-jlWR>V6HA_zGa(WQz*V#LEG5w8U7RI>y{B=O{uO6b;4z<aM(xg|BMyOjDT=_t)_3vhXg%eF0S#BE}|xca-r+<xUl#^L%_Q6@R}?b);eJ%W7v{Fv@?Dy*uasso!?>Wt(E+sl(XlFR7Iwq3B^MA?#PMN_jXo=8Bm_T|x$cXWFa@o$1x5mA4aD>9kPalJ@QSFN0Q2wpWaM77LJmRTE265Y#pVV2Sv=j&9#<%$Qhgrla@BZotsiuYTBPHxvNaF0oxsiJDjaFI#)N7kDtBa*!$5tq=9;DamMB5skm_LQYk2)lfFB=H!6sS;&goRr&$0Ci`8nQlP~1ls4>9a(@2&xzL{^LK`Wkk~l<Pk_-=59sc4)Uv5~Q51177+**Dmj@39|Caspu1U<Giy8du;PmMu{PVBwsbFEy+f$3>Z59rRN#V;d_4qnWn85^(Chg=#<cm_lU`nlVyr`vAnWR(c0kztGQH`1sU3Iv&`&EEv-3wjGQA#Yw-=l1uoC6Zh+<udaT!^jG6!fER0cC*d>XVHnuow_bFc5~v9@7M5iGYxx!{cCxXPQcjo7n%nXe^*;hc|y~-;b`TlN;3vlcqOn`U)uAJxw51y`v)i`Yw1jDl0I3HZ381bN2exyURD{FT?-5c=_Q++v9PFvI_#)p6;9ZdtRiJsZqRnwyu{-G_!a?Ig#&ma%8F1`_Rbbu`{7w?%4)z2}JDxfg!lL`?sJwyRxP&z)sS?|B~qRE8GH7RUh_1c0jGa-2h%c{Th9Cdjn2Tx}0BzFW$X*|N8vHxpf<5X)H{+?P2P;UDWZ{#RP3SS<xN;2HTeh8onmfn0PTTz%Md8`B}1xRVpc%cqpRi4KWU#C|s|Kn<Rub%ctXK%kqIxrO)zo?FfQYbbLF`STW`yl{$YJy5nn_)=e$o6LvJKz1*vp=^S^3HF4~>v}BU<z2Os*RtB!;?8->bP6ys8$~(#W>GOd%^oJ<ik+GjX8F>DaXLw^s+a$PCF?aGkF(PqM5WRZ(SL(!_3QBAOnzEnsTpSjKo!lEfbKUV~GV%t~#{tioe+~7olT&UR1wB;Qv}gdndNLY5o4Vs@vUBS)n>`I249ieYs@gdv5Ih?^^K#eqqZ*8!L%c}TaQic6qC#f>rU3!cC=|YG8UWqs%~yd2t3qPl27QnsE{rj2I&6eCj=c9sZY(P3L*is3xHUFwO=2(hMQr4m%ASXGr!`6N#GM{!kAc3i&NmS;k0iPu3qz@Ox6Jnv9Vh9B*v^C)Kq8h>rF|6>8qY`u@_EiA!ptT(+OazgVn4;t;glv=ZXKGe@%nh^?Lce?Rqm^;K{=EGiM=NLC;3!XwaZ16UUzydJ~I|Ge<VE?1=Jf_bJiD$FD~AF2#GA%Q+ImkyteO^(24`Z9n#@zm>BkTYiVAt8{75dJYxoQk=US%d~5KXf#yJVI#_WE4F%1v;?XO}oM2qFN-m=V?o@uY$ck&ne#8{Ob=#_j@~*v^O1?hn<dPKJjo{rRTP>wWBWHjvh=0wGp?FKvqOs8|Vpg^lPr2hEUlI^pBbWZ;Q`%HpIjzg!qY`>MwZlQa)%~PzE+vC7E};67FGW=<d(Oq9Ze#h~GKo(_4~d>gI7-Tf5E_37i$*2MF-Ll0N{K3avRq#SO+<Ct7Wry#!nnB0>1fA}y(*Tp`Z!t<TSWX&6)jFQbg1I-Y)j$^@ddLr=<=~@e_YIuslqYC5ITgtP1xPO|F5^Ag`X9ziIuJ_jp7e%%5E`O%Vaasm$H~;exxqvQI^3lVXgC4@iMZ(V>-#kos5XqP5W$$uywEK#Jnw<S5VUlcPBJtsVCX9UA=jI^78FlX8Wi<G3d%JbQY7SjpGYBTTCk!w2)PvG$-vOeOul0bD~2bT`)B>P}mHbuQ0xwZU^vAE^OW!y73K|)3r?Uakc&IR<b9wZX)6Q983<h+Aw<nQx(<mWC)kH7UfaCb*}Mt(;^po;3BOc2$gCh>QQ2sNDM@?-k^FW#&l6ExOgLd-=G9{@TcjKW(m=e``z<>#@xQhYm!HxXU-ezimDwy8E7ev`swXuO=Mve0U`=TyA^wSW*&x;JyZJZP?|Y%h4w3ABenjU*rl=7!Z7R9>hW|F9KAf?wa#2&%>Z96xMo1)9dy4xd+Cl^wT@;2X71SFv};`Rbk(Vuswqxp(^i0+6?8u*wi>M_R;kG&$Msi9lsm*@`_Biw*Yswcr@yTeC$5SzkL1YwEjS(N<#jWlzCd6``*SmZ!FLhNFg^@eM8s;|#K*dV+wFob@^ps+e%RxP3uJ~>9sc*1=kL!h&%^J}f4F!%vY?we#sz4xV(OTVK1mj=4ikYB0^5oeqdhu+>-IA&PL^n=yrN1MX&z`+eT<V693qxd-#xk)uYw&ia3lunM2_^XB0~+2n#9wBb89si_~{zpZb27(4$|(zQ!P|zr&@&AW&#oVxK(HAQKjux($UW{Rj5Zo2e4nrd~J`74?U}fJ*9T{*)&vx``+6-l>Hm)3^lFRkW^c^Ig~BHs@K;z|Iw@}>YCMHKpll7W8h&#C$6Lmn-PhPG-ho=kretI291y5zD5UMTQGprSjlZtZ4FLYDk@#u*!mjqYb(Ed`}!Z6wzL%M79_3au79LzY$9iSuxOG<svs`3Sf#|#WLvt4vh*gQhwF92WO~kG0~LUmsq2((BkGHFmym+Tj9UC8CZ{lkjcclI6#WY_E4C-6Egl>(6oQYR#t#UK!>7Z)5ct=1a_GQoTzEX`4#d<|-0iQTo9iof?l}U~zu-fs5+Drdj>9435{`!(h$caa8@dp4ng-pYmfQ%^u228q|Fm@ES?R$X+oF3ew7rT1KSQdJUep42Xman}A-Zf%5u#16fe)*gGa-?eF4S%XP<(NNWK}lXk#Qv$d-#~uLR=4R=!~$&e?@?<(kM%R2YHMv;!%q{_N?W4k-!B8sy!j17|tr_lFBlL%fL{LqBt0=w2bvjWpBu~V=LM{j*P`68yGMF>)WTt*HlB!1bSWaWwvh8%tv7;O46{nhUBN9s1a(~6YCVT2{bWwFtFD|+`fGZolCiG1=>1Sn++VB;?BypPH(IBHmBY831NSl&%I;$JB4{Nk8)OFGMRi}<~3|(Sw^?+@n88rf8gFTeLZotTMWXq-XmkP;+FNVCTU_{9pTvM+`^`pGChFKGD|dBtCD%T0kDOxHqlMO3KZ$z>Cnaf?hnYX;$GTrd@wR1V+hsJOu@3AbhW0gqD0+W`q`_%J+w4R%gx>C=<0fWYMX)y56e<kN6sE>sLz%<@eKCow%hCL8|yt?2Qm{}mj%Q)x+&_UqEjT66M{9nw!!->n$dgTdRZi*z>Skx8Ykl8X0a#47Jyeh!K>gf;87<{W{E;peaH4lXsbElniD(JeiimKSk`SRgIbr;bNfOshhB8^QL?D4*js=ubA6v(?XI^h9zaM{Xe(;$s1)Mcxi8qPn*e*U=!h6LU@#@oWFWFBw>jVn!H7k@Hm3dx^(OLu>MaiGt1zP4Doh)<HxIKoQXUt&c|aZdd5ya=p`E6|Jp&v#@DTF3XJ&+3>OliIzdjzie|ko1;H4`T>gvVWTekSnrO=UHPi`tIq(+{sR4Uc$m9vUUY&m=E_;T|T(h<j`Jx+v`!s}f{dpnQe025iveC15X*Fp>x90vAy>yBFGFZDE1;_u};hapvoPj3rRX)8vwW*ddYK4&EI`qqnOLJ~H!7tM(_75V=s2aSR-Z|qZVH3v4y2SBxjD(Io1?hOCX9zd7TR{ThKH?kFas1q=<)kzmw^Aoja{dz4uNM)Iecx}*(KxWx3Y>)j7U!A?Vc>Ry?#k-&1ez;Pq`nnJJm6(5m$y`ne@QjhT-7-ddX$qyFGxrAWcc;%phE>f+hqmD5*@v^M^A9xq-_I}4emD=$FE8I+j&vL77gAHmU6O1Io2|0y>j{i3`!V=;)y#ctkJY9Uzs_%Da>{cB0B{&uCMXG~h~Z)PlJACt!57W9H%XxfB%cUc*&Wb-JiGca#HneGDkEZJ(+r43@LQEoI=762TrvH#wZr*eHH2uEQpO)cK_6>kz-xjYpl*QH1vY~CYH-!`EaqLs@qh<`1WYZ(B$~wZ2)0+r#q<=_QB_4-r@Nm#;W)rEdLqpd#?$Rg{BWytx&v+N+Zx0@F=eh=P7mQcvA=S4x*{v?ypxO5^_0(8H%EPqF>lfgzjZyWC+cS0U5skRSQl<R7w7=8VXVohEEDo%Yy&PwR-f|lF4e8pgO)}wJ*Dbldl8$JcO3v>*+JijJORbTqmYITVHa<I2(K=FczgEY=gV_Cry|kz>Ej|$0B^4?H5cRN@}tGV$pc3b5eyE2g5eg*ZksQaV#L;<@2Y5`)Ky`Ap1=|;CJR6R+i(hP&76OAouinVwr_gf(o4;swU!@^E&`cA*3DJ4(UGmSvT)aHiU#gxEIP?+^Yq*+a{rOQE%WPkNJd?_p7!ETLSnkANQPwkLNGWUYJy_a<Gm5eEHVm#Orve6ZAen9Ru%I3euZ`*l=!(GN5S(@KML&ob=s~}jjlIEL&NqC=Z-)*`RoY5x~1q{kW;08?H-qJtT(L?mbAJ6x?<@Mt1f25Ij)(<S6VR7B9dQ&qr5JN@q|-$Qd!DhT`3Mkyi?94EwMQy>9DR*`k;G*XrNMRD~kBUA3i~c&61=Kn$v;^E8JL=SI}{m+|t_UC|;fa@OS38cuXIj>oS~=<tx`5HibHT88g$UZ4guvnD5vuEgC^8VAh2wT0Kv5!gJJ7b`5D`ll;YklCl>#ol@BQKom}%-VrT^TNwXRmTwuN-;36owHhuOnni2JX?MG?^Kv^v%3VD<?erQ&*4JvZjW7Y>keE@KP|fMC^eSSb$gZt|VRwF$k@xGA$0kk<VHb{L!devRx0r<tw?A92Ls?bj0DGyZFCtSeFJ6Ri&OTgT{6hEk>!07ey;8jG$*wZ6N050|=_Y6e8DE&|x!dtkERc75pfA6uX0^s*U98nDb^r>h|7pix>`WF89RIMr-KKuMjLO6_$5chOAY7|jC1yBMUJ3$T6DxPS)rxg@mq~dqk|W}YpiCzGR&+)A1`$jcAW*fy`M4)JvP))<Lq05}d}IBTBxNg=>JdP>#@w^?$xK$2)?kAm4_mC{02xT(s0|&X)?g~BfJQH)E$p5WSrN~9BEqNOAFn+)twy@PoyHGu|FG>8$Q>2PCFsdi0#i4wH1hJN?_Y1VCX+xvn9O7ED$IB3`H>}LvP@?;S>mWQdmMTqFFPJiX6I^8gLglFII&^5rK`8Q=(rLaa(5gN*n6JaXv;Fu@>}Bc5S0e+dD~-1_g8uV6!)#Icaifljho6laGA+bQa&Dae^>Rr(Y@X?JcL?aYoX5DqAozWYd>2;^JdX4jRYxC=?Rsiy3!YlrfXJh%cfXVQMshsli*j(>#@@8cZ$bV1f>AaiD*AvQSq_znx-2!X(HwHd!-Owz-QHPz|~4LWeMP(Nh;RbO;*i)pHz84FH&_@dPWAArdlUH(SfqI9(F6W?^cT}7V~X2^TOT1C3!x|7f>o;JqTPAv9AkaWd2o)OU&yizTp;@Er?*V<``54YQrol$4qX(SOTDNa!WkN3H;h&Y}XKBuwky(v;u8im#Gn@xMo@wYmcDXiVLp1SVvVv8^wf!7F0zL2xap&P3Jfc5_6(XmxWiI`#P&0T8;>jK=3AGtAQyC*q)*eJzyytl;CsU^sjspWK3mXWUX8%T2-QHo)qAg7NH?9G;6pT<$VzBiZLo6u2Z-%H<sN++gdM-VNgXt@1okOO%eO0@~Sd*<$^+IuV1(He}pcvql{R2aV(@9;bZX2wN~p|EI~rcJ3<Z%QJlhKmrArqoW$sky^q*RPBTmM6AsW7X4zsPE|Vs+Wl_|)9|#pHI*jZAz2Gn>YPoKf%G4CZ^H%CIASd!x8%%XbY|;{|)7R_t1uJw2cV_$*E@Hw0v#-1U+tB?lDUn|2H?b8Qwn0;y{FDASRi$EDQN0?*6?3CGZf(^0jd5o}nN=P6L&E=xj#n(?bnYIr(1OZD%ldLiXjavu!(m$@ngedNBfb8+KnU=uaWn+=&mItU;XWMs-^oAak4#Phl7WbWhWI<R3NR<#ZCE>U)B@emyCHUkPE(4wA-CNC1#9W)bAc_jKM&fE#mks@)jW*m#61T_4?3y<@owR~$YM|i=-fO6aDg~LokOUu?y#19cgI^4o8aH&?L|fN?NgXc^P9Z5%Y`G48bY++ld5pA$w7eO+&?>nfkW3Fi!N=q<`%lT@!?Lm;h)jhSJIc;deW$Ip_=5_c0hvrXN8|GQgu2NV9yu6=X0YK#{gCE5u<%-7x&Q|@sm;tN~s$3lV#LY^=#po*VO{Iry`ctVPa_!(+=76daL!K+GfLEEpn-A+&1V`O9zboJ5uHy5&x9h5A`N$2~&4W=tQJ4RR-~JBFpgPH9F9*+o#DtSzu74X3~~VJw2M+`8l<3-S3_pg8dHYWqDf3YB4Tr;A=Q^peg}T0q$PFc*iev{74Vt!aCI!UrlU$#GT=Z^??2PJ+1V`J)#r<i@J7*&_Ju<p*Vyv<#CuMPFS!v4<`>Lr9&{32JF<`HSIYsv#)>TDL*=Dr=nE#h)VIpa93r-NQ@PIQ6-azt;zg&4WG4YMk9yU0p{NA;?~$XI(buX;GaHWlJIyq1ZfomSX1dpUZ^+F1^OHK!{@3l)C9ags45S-=6!<=r)CFFTsW@w(68&Nsix!GsQ;3>?*^KVn?$|FB0Y`SM^;o5HVO{`c8GyS-MQ@MUe)4$?X<6MBj(AzuvL@X+X@|ib}Og{a#qA+G(|95r$yr}^{U>s_X+{)Hli(-n?+F#9g5u&`T+lUfM^8YFFKph@iPn+zAN1c3W1+eC&5om{!HozaY*l0S3A6RJsu%c&XYj$pw%<5GGWUAb>%obSPfvpI7;;^q*2etP~3)031ZfFvd`8&Dbts#CP&<_%(Tz^hUS*Z(gt1!q|c~L7p}M`Tz>z`$#efr5w9~=6@r3<Av7&vi0G`jm#1D{NYVtMA;hLsBssRV+%f9;RkBEH8Yz@uHOF35hzXUTY+{(}5^f#Gr~We!|4WCeZH;6f0Zrt61VJClHEBn@VD03ou15mruc-E#`0uEq7mv6bnxHFFpBL3#RK+s$F3`Jethyyp+BtTd4c$03-8jkbG_j$3%+)Y$_r&p^KklaHN7oRRd$nEn(W6tBnE(=#9V$>TeVa7U^ZpTBWABDo)rTM&1=yZg+^{5>g*}yDylfKDaEVbjA`Nwb*=fVRIOy@3dpyuzn;_#e2Y9#(HGewe!A=u%oZ)Be8FitytcmiaBa3ie5wc}#`4u@ExY~Ccp_oZNIL<HLmJT~p7mp=d*ttCsz**?jSq#8uj&JBv3fgKuiR(3dP8=IKFc9JMizqB7Fc@L4Qo2U;cGG@Rb6PszlOJzJx5!{X+d4yXBjv<><FSTVwe`8KQowLp2EwvWM-MpvsSE9iI~Kyg@fN1lu|YWJ^?G*08AB!#0~m5t3v?Q?3vKEQaT!<9W^6Tt*Bb|DYyxA12?%2Qi^w}wZ#`n*R8tE-;iMb(rm2c+akgpGnm7|4c_+<@J)J(<n~`?Msj)7_U2Nd~HCwM>AVR+@0!Rwy%qU}WO5=Jws_RZh$HS=(^HA{a>;lrWJ5nH;+)TCG(yi%>1nqIra3KF8f-Tzum#=)f)KpoDPl;KUU?xyju)8hrO<S<C1rRAVocUEvFhHEPWB}Z$ngE7>pGd-7)O4T?Dmn@Be;V)XrX)Whz+vC+l60|Tv+!8wbkT7~vozLRS05&79%8FeT(PPM_aW2O(OT`{`^txUCp1u*AMv6o9Cg%uIgV^_DoR?SQ1_CS>Yv5Y>OY;8UuxP9UzV&A<7ie1Mp$J5W1;Yj+m^Kir#8-$#lcsD>9~6694T-<dNvq5Cg40`K2KoP?7)~eNP)P85lu-LAimI(g?ej6LFLbu1<e9gG(FWxitOph$*HH@998V#b=QLW*>Ay|LFDeVf(x<m5yX(Q7ncMHpuFu*Gv<4J_B)f%&W+j_Xatpr_REC2o4Uqcj<?(WYFn;m_(f<7o!OEHp-uO3n8HOBf#^3O#6cpFYE>BtJHab0)3fXo(s8SZ*o2_QM1vM2tT-!YLx+C5l@>n2lK2{N#~1+{hI<tIrb3<NJZJ+y6X-ZW5E5uKQ)=Pv4pQvZVOs5u)q&I9nrDaCyOO@!qiGUr)m9M)6~8KlCxQGGlQ`~geXXG2mgv?aneCl2JbE1N{(D+FvQAIhyGDQycR|<79LBJ{bIhgpoMl{;vqN<0m?;)aRaJnAl}HkjWD0g1WGDTahr5#t_?;we@~8&>fAsXx>7yQ9d&Ig6MWg5{pB)|jhu~kDvyPScZmijHbqov7eC7S(rSZnN?0W<Ws(c^_4CtK~4bx@p5cVfjmk)KaiZ;&B1M`l1C`r|=6K~v;)4|i{I@nJ~)4L<-S7U|Bbd)YuMH&<49yn#!y)))&lvH{WxWh?_7NvJxm3TbjeU6U4v0icV#}CjgcPSpNbB>sRvMyb$E6D(f1BF;dOABQlvqUbdmJnmWRu$^9({87$)w<!>JKT6Do!p^yBMSX}>t!$5WHf6yLgd5c*~MFq$ncR+rzo-{ehpOp;rlmY-<jdKv!6!nafHV_iTS3A32)Sn(+fm^Xx%!IVe8dvOLDjPN3Sk^X;ab(lh;p5rxjQKy{vRH+WlG6JiEU-ZuGBaXahsu*#Zybf_w5U*W<iO5N@wCoza5^;@ncObJ61r{-tzS<rh*_DH9F1m$XFTysY{O>W-|v#CyPz&=9aiM23!r-<T39zZgDOIrn6SYzzH%fb4kHIuF5e8|md3hk?Rv*21HM8P{m5&tLB5YmoilPcQetw(q8w`xiEMe`^oy<mc_<gS$u!5iRildA3RRaNvDzvrW=8kZ@AX=0w0wlv{15JykV6Bk~VzIo+i0%-$K4Gfqmn$s;660A&#+YL}5q!<AG1VCKhhmBNB2p?Mr3z&2M&Rk{=!28Eb^bp8ubCIYiczorsEaSpZaz~E<GH%6v5eRkv*ohJQM&A8U|ivnA}{`%`)iZN}Km1d*8WwlvJC3{jNX^kZ2gp(Pysfz9k4xM15`0nEe#P0$S><gv}+7KWKf7Ic&dkuS$#wb%!&PJ-BPPXTgo$yoM?9dEC74Hs?9`y*1#4QBk*=0hPlXd&duDg&_Z$Bm#;OOE~!K-)SBsy_1JMw*4CanXG;7ZEDJICE#(8`KCk2|^3%6T@H7{Tx=4BD4~SB=YKbx7#XxJ7ccltXcJoGq*K|7igNmwS@vcq`S*NlQ0aPSS~k4sCbag57_okIf+QcF*}2Mwu!D7M{NfbiX4G(|J{}iAOx16tp^j@1g|auk#d1q}vQ8?58TY%7>3FkQ~Lq!BHB3z0h?%YJXx>y$uaGyaa`Jy0&s+cy-QR%;6Qm{mohU_T7hbtDk5||3Y-_tXU)1j&hYH#?w}V+MasYaf9x_%Mb~PP2j6#8rbi&1De{&s!M*R4|tadM8Jb+WA-3{ktj{>eCw>kG3^3^Tr?`z83{!TdR!8B<g+itivmvZ4U-E_--pyOun_Bsa;j$;oo2?=mB}X2Jw7%m0#kllr!e((uB)9}8W{9D<ifVuD>4#kndM?#A)XUbdV6-*4pMY6Yd2rUCh7{{yvHX@9~$;cDQ*MKc#1c*<H+f{-?hu-v}lqk)m0r#Y#}&&y>Na=+G*_9Ma$wc8_Z&WHKPu&W9%tRQcxQdVj4daq?;2z5jf?hT%TSN?D;|;8^$|s!S|1_vP8m~L>X4O3v8s!Q=t^$wWQ}^o`G}Ea3_hr4C{1bMRA;}gxeopyf?1I_9SM6jpAa?IfE*sQ&C<(t*fLQZj2y~Up!BV?J%a*`n!jnf72}Kp}daEj!F#PIz3&y?0mCY7bb#T>rBP6hhc`sS!nsFZVHS$onw}H>cF2_-Cb}Z`Rd2Swnz0EaBdTT>kM#*6V}_xb?}c+3<}6wcc-$5TdIQaN1a=h@UA)}O1aG*I6=0=u^F^TlMTJQ_I{ePRdl1+Cnd+S`{=$`h2a}$hGUyH`facmwRG6c;`tYOS!fIHK{pF!vi&#tS)A_qS)ASzYITtlOgWlg4!`|anKBVO#2(~5ho*fntTmbY#yZ;x9J5V~HoOGHlZ=T#0=<gR3KfXf5e%ZeOG!()FuWz`m*Fg-K`MVbM+nrCGLeGN33UGgWd65bSvu_1W$yx<&sM1@{sFCTLE6Bv=QAA)_I(SgJ?8@M@UVvg=GZwHbnasJxEO$A6mM|ttpxDyI<B$`NIWX2S`B+9yW=VEK)SmdrmYhE@s0$D>JBIr=~ke|;tHaCMirX~yRb0mJ6k_1)){*S%Eq^DgX$|A;(vIzq<<2mOA-SFkt>G6vD{2-;LYJS>;LhF_7O*+glqW2?D-+>o)5zthU+Ti*1j+dDxH*L-w{UX3*e$*+U5nXls#O_kp~qFBpr(t?60N^2pnTfFA$?U9;?^oj%Gslc=}hvQ3M?tz9QTo|NgA5p8S<vM&QS=<2BOt%g@t(C?OM$TQTX~=T35<1%t3Z5S6r^CI?z@;Dg8NPREg1A>7AP&uC6RFfc!A?WXh~?gYb_QHTy2W75>+6zB($<`r)64jG6Iy@?iy%2-HjpV080hY4V#olGEOn*Jr3esgfJANzb624fbMc(6GU6jpV#*xnrnW#Xi>@ZG7tE6leT7Vi#@sZ-S|Zbuj;BkL7wA_X5>U<Vqy>}z$O4~xQlMb^7JEtGvRjNP1qRK5VQmCIg=HR0@Bt@qt+TR$D6fwJ$4&yM!#9C!x()M0)Dkxc&^zS~VKECy#S=*?oP+;r=piCY)lL(!!VyXsn><EyKeCda+*`RmpP+;!`ox9&IA1!pxRR(2$i<WC$azMf>OB*lecN&q=bjo_?_qD#;LvQXb7u#ig9&3`uWq;vSy6Xcs4weZiqF2DQ!zDKa@a~odT1HQ7_47_o_d)l&2-4>;rX`dZb$Ak|Er^MOwqI^s8_tnMsug^oa0EF+~y}o$y58|%`O&;4;u7c*(z=oYD-Sn|gdr|c;Mf|7G@DPsbCM^i{+0oxv5l+2Tli-|mT8VpAE}UFt=e6ip+%c;>XdAk$bghnpf3@AyZYB?8Z=+t^9a17%<9@`@lW&6rE@N6x+#5NOQFY$Io{F-s)^w+;6Rq|<(9c&vUooni7{-J@9=%nS_O-2Aq3TZCJdRCB3k<bkPpboP|7sE!Z#*puDb2n3{B*{@&;F2<^@U%kQaL$HiEaEv0#gzkK&&zB1l^0>a6PKfeH{p{5tBqkc_MT{052i28Z3lko5LZV7-=%{jNYb)0&O>nom63%8<*8SB>x@(EI|*54vXE%VjI{i+O&vdU(lhD<9j>nhf<wUOP@B`*-YBqq|>trP|hC-nM>MAlEG$y347<aH4$5UMuL@3fioO<e;vSCaCWdm*i5subm+C&uXj8=^@gYJ0q=z)q1FL#ZDP~24R2n$vZVlzwn{g`SQiAlW+5ZnZHBVN^$MD|Y;n!1)EAwtvz7WdPK|@BkiP_`%~rwl_TAg#SFcq#X{3@pv%x>09{txe;tPlf(0EiNci{scE7T*|>&XCNpdsP)Ex~^Hj6Zl3(`B#>o_UPzAeY#;z#ly0Si%(nZ?;S!0*b#onpMdnl<c^{(Dz7ay)J4bGdLa+v+^rEzlJ+L<4`x~dG2kDw>5VOg5<6E;Q8vnOHKr8!&@<RIj{Jb_TX?W?1`$xZWFLo(%Q$!o5r^Q{1h5KQVJIKwDb9qKCg>{U0L~iO6**aIWBE?mB5{LULZJbOq;Mm&OT`qvWG(*%<YdD=@a-)WCDswtaR8zI^kHt))7J?6kTF|*l9-;>rq<4j?IoygcFs_U(26wUcYAv^QW`7<K2$vdMG)au~jFG8wrbp)+#m}ENTyv_CjGooeg_OY62=wl0rN?haK9D)}w;D+K8jEu7GXmzh+)mzwJQ=S3Bm=U1jyHSF30vjr*Z&81ECFT(9#SA!My*Rmv%f1G8&h6>%WqX%Jp-k+EBnRIFAlj~vxEA)I^Ju*7((y*?mIVDq(Pr_IDRyFzkyl_craP@nV~X@4={VayuZ7_2Hu=zVS0=vjhwHLfR{VAfUE=(ZNmUwfB0tx`|%_}*OKf^O^4)*pM%zHCnOkB7f!bY=GLjjpyDEleYym=hfNL+Ixz`vPa4)poB~l=jLDV5tB1uiSeB^yeJZd>@l1vCD!Cjr#CO)Ax=v@J7Cw_9r#fAZTaL$}ylk9e;d(Z1k~x^(W)2WKu_e)a;Y};OESglr$^X_7STh#R87(pzJw;<e`bko8s;n<>?$(6K5PvGtNOOU&Cn;J@p>KmHxUlV7y^3g$Qz^cb&M(WfY%YRoq=S)lv62;cuq?kxE){HIwVsXozwVYP=FxN*&xI@TyxGFSS3$)-}9Zr4#qNX9wfF4ofJu`ZUA$Q3hSV&hb!*g+i=5%S0}3HA~aVBO$))yc6#TmV?7SzLuhhSYz-Y+pD0#$UT|Y(?d)z_m|A-gMNDrF0L<8?2m7^!FFQsLBt4kCBg_z6{vv;PU}UN5SNxYW<76be|5PECpy%Shum++#VXPM!DM^vMx<oy{5Gmm?4gD8A!;X14g&|``gU>)!^6}-7kP!azys*=p(u#B+-c3PA=R51*pOF><OdZv)cX*v(`-GcaK_QPtaOcgK2EFaw*JZT@i>D-8-81wOt)Vg?ZTEc!>pF(4UeUr!k@ri5|5fxl2yxMUrM^S{}!}ejhix1ZjkKnNfrJ~lVf|1JD%V3a^R{C=eb$opF7Q)``)(g*<Zj+(e-dFSwDu^_?l|aQoTT(9J$yO6bU+jWT1f40kVDSMWAiX2sG^^ilY{4vz1yRRa1(*dut6O?u-7@L1csp$o1WnC*p<Ll#l1RicP~L=laFGeB96r9j`VFP9L?6@c4KxEv5Sw!}>j!)B$5G|LX~E^1t+&s?sPlqDLD!&pp~O8d0T@Qg_k+;Pxn;Il++5KqB$wCdGGk@SqW5u1LOw5gJW|a+up_QYcps*vhJ0Lz7z-LCE|nr*bClmME?I2(Q*s$z26zi+_uj#Ikl+T~tylXbY+8A(mFE9(bd>zK2(qc86b5Q+(1!d)Nk!0Q{>xK~^0>&1U=$ZnM}3C5?{K?IE!8^x3z8`ue#qKbcaH(9_>Ky1^JI#IO0wWD}<g<egPJij(~>bS>EOV9#HQU>(Jf^7gmh%)aJdS2D_|Vow1oSRd*fD-SM99gBNb<QiGtx;syO7Dio=_wq~nGrGyf9VrI1hb+WY;#-0Dv7vB&jcUeP?nBUyWn^62U)B|;7YDaEOAsL9OIF>GmeyZn_eC5@hW&3JhXsdcMuIo-{rJy!mp`3f+FmCIrEPjQ&49a}UEHXz18O3RPd(h=o2^XZ1jwAsI8{Dtc-qOw1L12{-y)=vqaZ77)T>6lEL&BZU`si%%xyKjPOAA7encOi4&~XBz53k7y>D3CV42h$A#_&h3`w5p(s#$YxP<{e382Bz_qsVaW=pul7f1j6oTeI!rKJY<XEekv;Py~0c~e^NAKClx)y3P3s~^u_nn+;R>lt*;gaU^*zSJ&@{k_+v2X>#+b*I@O>*d`3*f#palDcA$k+ur{uZ{JGYjaKEK3XiQWC0=4km~W8#d%;B_Mg#@T@#HU#k32&GuqO-U=Pk|504a8nz!FvWLaX)VO#t~3FE7lGN?Y`TFroY%}rO5F3+|!xGxJjzZ7x3caRXYvkKb2TZDGURb>=+2rb*RTB%P=JeWLOQs>mZb<x`*OK*$#BdV|As!SsIG!~K)nyLc&*=Y&z3mtT-v3vGIWRQgyi(rLp!@5)3Ge+QS!x~?^p-`+R%H=$$`od&+c&ll+Pl9V|tBucVVj4P**YFB0drqGMT|L<0gYa{X-N=w=Z3mtsiw{XVS5BGJxm!aFFcd{l4ik@|UsaWMrdFkoRbLt&hKKdRd0phXY3CIEQ9pg^`n5i`kDuDz|F|=l%J%d3q)O*o^&uU($WG*DJ85(Nh>XduNB2jFiBFrPpW6uuld!QyB?3Yyy7+H#2HRe0M+0YCGk>dz4O@#o##V8-DmMvN_V4Ez+plc7B!YQ1N5$y`N=hlC_3KK_`IWuf`@LM_PUn9&g5kb!2;uB{{qYgjZxwUcs{s14b64ZDP@y6;`<F8jo<sFJ+3mn}e*ZS@^kgXZ+RL`SHxZ|9rH~SqO~Oa`-s;=Zz*u-YY!L&shYy0Vh@$Jn3Vht1!3>LXPoRgbt`1gRrgC)aP9}C=)<!!eU3cHun-B2}p_u=c{7L&(h_<Iw_CkTK9Oh2DZh)sfwKNQvzVR^z+yAw|mxdzo#^QVeI2uHxreecw<fOE%F#TPy*Xz{^+B=L((qnuG;@?>BBX;DElZ?m|!Z4w;?1yq{l=a|%&IS+QUr|CZ<n)#_StOa#2@na66Ng~gcO`QFj51Tm3(U|qcRIQ6RNudA-oyLT+Y=}u#XL&1mu9j`s4GX$*(15)%g3L3?DM!`=OTQKN_S`LFKfw(lP@%W!Cd)sS!%Kqn>Q!4A4_M`pnanXj&}+k8xt{By^A0vZ+$3RA9f)RhY+Rzc~%l(%B*Nee1svf{oxbUckHvpf;gGIntr*(6J*6QYpC7tK|OKmeJrUTKoY#&QN_U~?J}2`F1%<HQ0G1gu^6j%VAJ6*Q4a^+v-TUW5?aTf)%m^4lBqV?9Yq@-t(}G6#|LlW_qULbmVEj^eFy*n)kziD1*>?eBOLNAzjKI(5Vu%qZ2gDz>`$h4_|^MMf?L}rT{-x;HVU$20S0DX5Dmm?o+BuK;WyC0gjuwO_~Qx${5rZ#=<g2kWlR&xXqR!?FA=m9muWD3I&jAT+$<|-3#q+(@5f*D_gb<-g}(5mrdc_7#XvaUkuXidjv!Dl-|N~;t}G?^-j}c`w%77d`oZgXr}jw;UBM13;~j~`Nc402_FMOnS+1<MN_X1X1GE<}_0OTH?99YpeYXD~KD`%^&R+uSV%6KTiT;G|=+XufWj$aq;E))$MoavS_B&U$WD#jP-mI3cNhUBqOe1L-Ep~x5{0k8%3!iPBdv-6ppzTb>=$xpS9jgm;VbAk6S_k(8u8}sFO@AAK<1VE;?TlN(M<iwUdpIk=-;u*{h?^{oKpZUa?C1BFF96t#^Y2s$-<hQnJ=f-6M&D-~qbCcbQ^lJj8GyAfL@(S**-TVT*)7Sxt5+mBkNGoWzo#WaO8x(xmiTzF(?%uVAE5644uy<;XM1hqi}&Hn^H;CWKAgW~tfnof`AZJLGZ*LV_mTNIY85N3KSpi033<xytK*~JMPK5S9;BO$zYY;Lg`S|h8(s9_{OUvd*`Jqoceq&3)Sfw;=E?y|pSQ|hv{8!GW{||gkmz_AB7t2P!t<#N2<H63LjSB<tij#A$FB+>QuCuY4kK0U*vH56*;vV2gR`fiB>_{G#c*$G%>u{BOLU^9KKmW&W;*whr&0hRi|c*0>jTx1sVoZMnn-g;o7yq%N@CZ1Or6%QXFS@8WM-w9Sy9bw)N_W2e0lKnL)~iAcJ(S|jXf1+r-o!q5G~}m-r@^=Y2(mp{x30$k9h')).decode('utf-8')
(WORK / 'edge_fixed_base.py').write_text(base_source, encoding='utf8')
addon_source = zlib.decompress(base64.b85decode('c-pmGdvn`HlK)?y0++2l0MZba<T&dkgH*{D?Pw!QUP;d8j>STO;E+TF0tf&~q*MOx_v@Y+JV;7Tu6A9i2$)CDOi#bN!Ee6#<{w#{R@QJjm%WRN?5>oRb-T!lDq2-9&a-70r`FkO8duhl^-fM@5ms_){TP>3HZQ`Z^}YzFu>^!SSyonYI`<B4WGTZUT3C6JRaulJeJjmUKtH$2c)3a#LX^;~QlQH!!pB}AqpX-(avd$gbS|ywssJ!+QbRpUtt_wNEDaNks$@YA)36F@$8l-NhcH<&=qyb(*5g8^7?&q;guTyXSgi_arQuSR)-tT3g?BJa=Sf^HdNkg%Zl$!U1%_mm#H#n!EGw+3EaQ1<CGj$@LR!4^4!-&3n}c|n14UL=9?+<J3rKS8>yN{$H<xF(!PTb^!QGEH!?W{STi73dy4-5H{_t_D1#iP&KHhX%-F{YjGJS}PEFImCefz{V>;771sj#B*L0IJmIIQBO#Q%#hl9MpH7gkuz^ROt<_&LsJae{|MST2%yBCN--NP$soy~vhU9#+(BX~-Y(YF0^C%Y0*nrIqHUDG$>r8t`8}RrAXz&Np6}%P7Wu=wZNnT<QM%1jLYfOIa0hRGJY*76sud3QF8MHWxmCu;wzzK?ZRNJev+07p;m1JT`rJ2$Lk3@tB}o<pk1fQG$|#S-h6hU;^5py}mg(xVgT*^XYI7kUEzY@aovZ^Y_Ex?9=(hT|oV~hZpzZe4fY`k6CdK9NAVpW0W{MJ+2p*!89%$*OCdAv0p6H>IKo`i(iSO0^UcGVW+n19$bIA^J$G2HgOs;TEL7KfLT^aRoVlL@Rs)k5jRJy;olW2DdZZg7G(GQ<{rF#_ul{AE+tL}+O6#V=vaJvDqenj{I{{N7c!Y<D?say?2kZ3a2tgwT{ez(exBV1pUywx@%EpWL7e_7Bcf=$--KzIEkPI39*YBu{@I#i0&o|DE5n(5d5pohTM&VqzWfd^X%-ODOZ6BnWOQGymg=pPzXf@?NwRR-fL&%&piAv9OXT!2o-eBRZ$4nkCE=>|4rx=1nfGrR5EY19Rsq;Oo-$7cv%COA@AyP`$FIi1vO&24OOBw(R_PS$eRmA(5GE&air16x3^b-rU??ELX7|05zX>}zwfiqU%+Ny0?!OW?ZYGWv304l1d=X-34PzOu10#~K;v+3^dW?UBE#16~LDLJMcp7NbV$^E`p&mx;X!o~ibok5N@aF35GI)FYGcaSjJM&0t*?oIx1f!OQzz)eqBRC;#B{<D0WS&4)Os&VbT38{3ND9$c#ZgPLvgr5hF9%?O?fv}Y`r_&?xczYb)6jN(l1zONrnky*A=0U-(;pp=g`r7*mn6G%I2LW1^IIfWQ`uN-->m=IeP)BhA<NMN3%eV&xMu8+z<W;VvXC!He79unE{HJ%ToP9f_Tr8Y4yJMj{j|&-x-CJNmTsTL1U0zcV-Z&}sDMj{B(+ClONYma7`&5K{;2?v5S;IqRpAPmM%ffd@$FSL>%F#JwFo47T4IY!$Iy(>97TtMBKo+RK*0%%<sAuZ2_kK#4M@A@%xG3Fi=JGE%RG@*Ny-d*74jJ8uVe`g8CR@lNxPAPEaJ7V*BaRxaA0`f9uJ__SIe|)#x#8g7Uquh0FMs5c4i9IGeWgF3kI(JH21<XEQ)aBs8vUO5wFK$T5WPky)%HXzWdhIgVQh#sw}{coVCD<f9#1oE8}!VPpO=%Ck!tnR!J>hAj;Azp0Bc%UJ8xi?VSFSf_K*LAs&-#e!^{FRv0QvdJHEedc}z+PflG|^iE!jlh-al(*7KqclYv?L_FUZWOo&_P|~u)!TI9|7q<~ae{^pJ?irAANnkQh(?Nh%=A5IGlb7ybvWo7R3r1r`TtJ8`$TD)6mW^Z@gC2Q6lrR}P$Kr(6uww8=3%7a~?r1aG(>nHj%pZGEmTw$)fUS7tVl|s3(g7Uc(t4C6m`k6!YDwsfcrV+MaXdcU+9Lo&U{^gKCLl;HTf$&)5}EQ6LN7o=`8Z-i$B|5VBvk08m<KUh(DkjB&@_$$>yG-pld<deHQGke*ds}nPMwKYWyJH2+mJ*X1(Sx-AQC`lygV<ifoX<c7+u6FBp~CazT>qp=4?uxnmKP}p@eND;|G|Xn(lj#G3*79*bC?l3x`;SMJ(A8uB=iqC@B%eic^1$y9jeRI>Ak27HDETz)VRG>h6IO+OOUN-~P-pnC;UXFU)ho#(IRNG10ek?79lfc3-%rz5^@C9(~0NPMb;Td?JKZ;vc(%1ul&TVN!>a0k<A?z&06^BLq)$?-2eIz{#P59OfVgi;aFAyF$GYSb9A=k-b+6r$;C5=!9;WPyB1c4pVpB+$=Ao?hBIwB~fra+|ImL3x#GY%3CR+VdT`|kyu-hAi|$Ao=V>?h&XL`d+p|G+_5_CNi7-xcm;}remaPfuq>^vJ)=m3jJk%hdM&v;t9>m<C2p&IA*Hpa+8Ty1C-veTSa$$K+WzYLt~K3QRB#n!L_QGg*h0?PHV`EKLx7TbuPWdGy?=9l2S{v+8G5qt9I@AC&9$>vzW7m=WehE2L5L}RvuwFp^@&P$#&~bgI*K1a@g+R1tn7><IHgQr3zTG#u%)zhMvqYjmcZ=>FdX742p}?(nMjQ_05dOT{`6ttC52DsTypBa^jId+Sg#&y9tOIZr>Q@x3G<UeKc4!@sU8RmmU)ohqHMWbrE#PT(a8q7)IU~Rgho}nTSNA1T=F(jQ8V?x+O(zgrX!4Av)tr~0Z<&L{^)@z@S(-+hSeY3ACAZ7jEdj>h{RHWMGFPA=wGb2+`B8KEC-%fy>^x9OLaZ7^d5v9`Lx0IdhX%$@bvIo%weQ=ShYETJ7DP!+pqLV)|1J$89l%l!;9-~qe6o=x(3Q9;b{sxPmd--@|E^?oO(sbn-@+EsMl_H)zwepu=JI@A`k?MShZq7H(Mo%lM3qCXUhC3T%8`i)>@HeS1o-y<jj<Xi@T=a2fLkZ?s9;L9&xXrtL!Zl_qw1Qc;1!_qs2~WhHexWT*0|ubLF!vveI#;9a`CYHa%-m&w|l?FC7hwN9<@VbyhxumHQk&a1=IKE4hQYpZ7NWJTMxEO|<;bIqBsv(4Z+8^satTP|aB&+m8qU_Rp8<&_M_=Nqb}F%mpYPH!-gk{uXaE=lesD7SBO?nhK_et-+cCta;)$mXAJ;Q9IC9q#Xx~-7$-;F$}8kH2~(O6LAk6FpZZg>frX~$zWlIEcOpsBn*h;lK1M+nJ0(y<b|_Hj<m#dkLJbUyg*Y?H}x<K&F<mhDGzORxivKG|8XG;>0rbU*!F(|0{s=#D|##(XFMFTkBla2DQCoM(xEZX!71$6{-luMJ(C;slBjSXQp#xonok1dmyqx+=48I`L|#IPX0rgbj0}MB_x^DMHj%SRO8_9O8AL~G-s%$`YOTVM*`lZ?7oEw)*CscL$QW-d9BGy-=m}F@LE96&jzaEVENzia3V3W0>=dO}NsO}yXfghukx$OIN8cX3K6?Fi&R+U8i_l(yO>C5VTD*SL5Wup1hCeAasinaaq$sHyZT?ZaQqnohj5va1FS1Ff)in!@$&*?Mou6wFPpiN^tQEpufyfd`R3Z@t;icJC|8wiG6E--35*p_=n^nT_vvx2!<_SZAHcS*E%SD#3jN9bCL#;pfO|vmZK@;v^uc<*j2UeBSgX1^=!W_4uKd#oagV~<$<GLG8NZRT@-kDzQ>26Qg2c?T}R_T43J(?J=U*%+fK=@So(G2`au?mX!`7_fo%a1B_)Y0MYl-z{M%)hWAyFZwn_@{^UNP5IrwL%UEfQl6a&NWK~wqK>&U{*yIY4+{Jvh~%#Jkf?3i=6nk6?$CP_fJ*0SVb~+faZk(`b0$hPl4AY^8guMlLUpKf(-?&0!2a`O(pu7$TXg(Z5lCy+|gF!1}l-ZspT*2L(O*kU_Pw>GsU`wMUfSbeU)j@Rui(dbR?>@as7X>hU+k@EV;}nk^sx4SW%FQy#d%GoJZINL7oK%xK>y&o2f8X5>9{sucj%5Ol<$RRhU+BBK1baNl%?zEG2xAsc)0dDT_Lg1M5A|!og8P*p9CXpbk`62Xo-E6_rm37%!K_EMQAO77iJ&%cXxjC@Y!6Cm3kyRK}mR=MLoQwf*g2I0+y=bwVxk$<m}vF46mzyvcYLOOl*RzuV-b0niXbZe>O3|NU5W;@iG={E9=UZ1=$=cqJ%yJ(W4ElUIrwDNr3P;$#}ASOkxP!x$vtaxxA5Q=tM^0XeJECp^IC`Zb$ipa9WMvNEnVe(!`t*wP~}Xg#WmvDnc6-9i4~FTJkVU5_5ZF<B?DU-D{CcO`W<=>W#(-;W27pHUHWWZSpj0=ETd<1<A%Rlw6=({hMLYlqmi55OaB<aZMfe;oJP3-WhkexMQ^?%TR{6O`E#wxP3uy4kfPQdUI51Mh_%+JwW}{$xAa^Zm<X^69rEgR-IB?@#+22u)Y=!1_<n&R*&{{tERR-9Jq~H3889gg(cB&}0msXK2Y)czk+fO;+wfZA%htEyQlX$>8fojKO9p;|WBQ@~>(FGl5P51M(*CM20Cz%Ce4><yGCzyJ~`CHAS<@LX+GsZGXTdTzx5*w01@@bYNX_A_2}G+dCX5YHw;_k?W;8DQ48MnwSMj+5e9a@zU=`x>Ue=CnYx)+}<SRbVD~GXBtoKv8za-6--g+-cJEf3lb%mm;3C<`ZDN<OKmxJFQch}Xu@_TqSg-9ZH4EQ5HgoTGDs_^{6XZEU^P=paObq#r%ZaRX2iZ|6SWC*%_g?Rrx2ZMYrlvvnP){@EtWz1aII&p1tTyk>04RnHysi=xzD%K%~~pzmJ?C^rc&*Bi?4p8d^uGj0DwOFRhCL&*SXD)XLmmaSJ!vJyX#L^=XPTN|0ZnQ-$Oi=)25=MPE_*RWJH{2yiguT_f}X;V&J6MSZ{BCR^t38&fkMp^HtTiVv23yGGJG5)kmlPrOZ}EM1fLr2J4$r+r7GOva#o9cW1Z5yWs5Z?(FT4!}C5ItjgNS*ph!0rdwRA)|e7s;sh*JFx0aZ-fPTKZcU{b)r2~EEVw2`XBz6XFWW%MFp_Ljaj6UOKg33-2u5(kwMHj6kg&z(lS(`+WC%+4-%y}Dorc9Uh{}gRwW>VlU{!|Lw?B$h%7%TL?k?%}xn0duBtp!krn=vE>1bb)d`twjV^z-Vi>tTSA3!q~Z!U+yEztkrOoxsA-&-qu!S(&Nin*B<`{udDz8QZZA+C(b&bie7Y4>j+7K)`4xvJ<#`4c?D7Pdm8c~gJgmZ8jm_t?laxxBf!I=lIw;PULv@Uq1WW>da`FghL&)@&(knk-PXfbT?`!j*+H$=Ahgr6M4OkfRZU7>n0bpHO6v0TYs6lauw}^DI5iTZXx#l(+S{4{27w74?$rk;D>c1M-4>IAxjp4`A~+bw)Pj?N10JY;JQBzgUKZ6@@W8;1#r&t#pR4QNZz32XI=}3Y$i}F)SX9c$7X8DINGG9QWd+bD(oO`P5UXVoF}KW*tRiAqnNBZ%3H~soJg{lW!d55!>2x*XbU<e!UQFb6p!0SQAclX)=uuYExr_fs?1s+R6}dl;5rCWBn{`{79u`*TK(cmlx;3&Go--i7blnQ5`&mR*)&#hU>ENsny0s%YVZJxBb2;x04PmfvC<2SFHX7z_of}i&N+Pec-nMUj!s_A=uw|?=F6MOOg;eGpxd<Hefol#~2FPs>5#6BFj}6aj@}etfx39Py%S{Mw)bpVN{4EQ!Ij$&*C}JxGIb|u-KLyIWu^>Y`>Vg62Zksx)1@Kbg{|?F=bOKxkz*BzE;tQny7`S>F?A@sUf^fJwR7HjX|YkXwPA)`J>M~FcqJ>Rz2;UJ^S>p8r{BM-@uqIm5arQ@$tx_mS1LJoT#*L8qXvZ-i!w!6>YL`h{9-rLvZ@M^nd>{&<z~iA_%{KIneuY2o+mMV>ij|Zq%Ga7aLT?h8>Z6x$zqiCW4AqrfxD5+gs^(-bNTU#=S1XdsDxs68=OUq6?I6&)=Ors0yFDMmF0+_86XmZUAfA<X4$~Nf)YuqMyKTJe~_dt#_X;FMD)4b<Z$5R=GzEY2pr?@gzN4NUo0My{c>p=h6Zkf}a9{7vvc@f1^$TliH@RVNk=j4U%f*+(DGNqW;$VNDm+TyCp((pJ-SDV#|LD1obK_lzcM($-oDltrh#=C~-Vz?j6~IITjdwmwQ*Lw1TFz%J!fxbWtir+)rP9$3-4+wH?4=BOajG$c2XS7IHm}=aMQp7}pdCd>#TB`_DiC4NIzWRS5)AI4p=};WAD(UQAks58XPD>-$8Ae{?#I@a$X_qSl+`rCbtpV0%>(*L30}6x};E5q1^3s%6anlE|?Ny2Cy1MgTXoF@Y=BvA=<?&GmJJk(MKJfQiQdwA3+bk10co%s?I2ui4XY%MKEXepin8nxvW?b2L}S;!HTSb&z`7zACjd?NB@%?NPyCj`BJA!-&sD!AXjOyUO2on;0<Qk$5D;R}yNFUjrR;_Emxzr`M5b)2}0=9K+^*3SXNs1~iW(YV<W}ws*_z9nu$vhZ^lFIaAB?e=C|(T{yr8=DY}+un%8_8q5i6gd84{3{X2V%H-dB?|%|YW<4cPP1M-u0wwxbK<03`_z#Qb5^YTNq!ItE8)vK0FJxXga_tJPwfF0adygb=SYd;a<Bob%{M#S9#;X16$uTIVpBF?y_P?%gej45cA8)ST-wbbWZPD0Rsjy$zf-b`-SXbEPDgs%Qs=X!?bkG^)RJGGHRZp#eK~*Wv>8&z+U`N0qpO3746*0u7=(!IA?i@nvU4dSQdr4l=`0G!1FJ`vh+3)oZzXZP%q%R)E*j&rH_TRy-ruFKzY)#m<(2~O)G=22}xL2R0)o-ZI7UqeLLdCXEZi&uLiqE5BTU#(HV5D}iO}^sg@NU?~_Wv8Rq^{B0FWLBqiE^rq<~k7Mw5(<L*qp~;Cfl+%Rhn|!&}_AK&C6DYF)Q^&lO4{N%=b(F9t^`S#^TQ9so-5xCSf*A=0KXIa+y~ff0XmEoQ5$>j_`Q=O#)OZn?hVkSFtb1!KEKgUX7dB4Jb^xChU!|w?NmNyA5Eo+E{?%>4OeMR*Y-q7^!}h+3Jui4zFm<=kwQu=aPA>+Zuv%8~a!fH?&5c1z&knQn`y6My$Vs(=PQX=4w#XS8$Dw$mQT<lfhq6*x)L0SNni!|DHN-Mx~>rdZD#zCa{YYySUZLkh}^Xa*JORFTj@+v8-Lj55v2gi?_FmL)eQ_Lr49W#}HP7y;1M&$u-K`lES9XaWwLGMFU;N&ge$p?T)WCPkWMb30P~?{rjX4SE_A%y~%lzt@6pnshLhI0J!8W)*=U;#XKImmI=G-^SMJ@;XBNv6w6SF726g1QzWu8(A`a5Gt@9`MwQic-yk%rFFLTEfn@UB4GhjWn{Iw-_vJ;a4==4glkl$lO+!7#?fB*m5<B?UZ48A@vJd&{10cV?pJ%YonAFIvjRKIQK03X15KZ?{581LA+5?G^(uc5!!?f}zS(Z4C+LhN{h3j;aM2j#pqP0dnpb5^@VE3kn-d+4MJP+=Mw|BwC)z4=)7iU**hrzqEi_77;N^-YTly&MTRu!^VXyyDJW=5iXCJUKHa?q;c<<toUAWN&1c}R^EnTE=)MV7M=rI?Az6>}0^UuOVK%w0a%A%70P%9%eO&Aq4c)njc_$e9*bt-u|By`TzT=3ZQp9rai`b>gyk5@;t^$p!Jx*>l>M>soR<ZiX@H%&OXYhbpZ?^I73T8K?}hDev&Q9NVK~vVV6x-s*BY@D#`Hg5&lM6mIEGCRwmM)rf`8Y}3eE<av|M&O-`?2kP_zFIDapqe_`{6bq#)5?M*x1o0s?;FKEshFOVI?W|SwXOWy&1p2cgFp9}c2%yKxdODq68)UOpqWa7tLQQQN)w?bSGK?8fHIwF1g4X(K*B{xp^*3(sF6#njs-y>DwzEI#JB)m>DXA0%tU0BGUK)Wni#Y__X9dyb+uHg_kxEh8rTbe+Kn0p2B|T1?@2S$)U{(2bFOd&2*;{qrWE|-TU<4~FHaYJjU*L?pwf-7%m&r_Wq_iR>olNmW5k5A#V+w5O>6euJ@4+JLDaPy3(hy2?u&Bt&vX{z=tRwvap*5GOWU1HQ#JkSt*eVf4z7#ugrCd^}PE=aURjD&rEJea})1>`4f~TOJfx>1ne}5d2h?r7UA6TAbs72w#QpM})QmPkcpFZBa<&V;F4@tIJQmwnLeURFye%jhnFYcfjqGHgx8|FF7`&*K~RrArhLXY`kUB&IF7!%uEK(pR+EmLzogM5nim>7svty2CN9^h}x&F!QXS-~%e{IHfqIal??h)MQgNSzimpA2?#gC038m7sL6zb^xP(BvnzZ1~+)4|d-MB@mxNl_&$$?h_ngt1nwAe}u_w*yFDf5|ml}a$uS?HBn<bw*2*^14ks;sMP6alYUT*D$Ar-A&VzV9wde3RCm+zTn=XV&rfHUx53@b*~Qhx)%)P~`qRzZp^D<_TnktykYXeKZW5O;y60m_Ro}4t1#!LEtx_JzWjxR9@D|&;90qSM&u(wkzSi3xPO2R~`|+fi)%{BHE1yaJk?$lKbWj#iD{`$7XTsG3Yt7$KJm@qY4;Sn87(E*<Vl9GEH@EahvPZZQ9r5Rnk@(V8$lvhVi&k|ZCx}Qg#q^y^&43|zggudgiNUIjtg_<FZ%}x3l44IjqqB3d6%e2=rnkcZXmbZu@kC;w3$By?*iANUB}7ypol8EIugm4kgl?B(A8Q?)?2x9aOH-LHHX~FbeCK=qt?&2sXBg!J<fkagy?KUGY+5Cj`|wbI;qeE9-+Q9pD*TOoRQhv{&gnWvW;X<=Ya6yK&pj8&rs$sCn^^JQ9s5;N&K>Bbnh_NJ?%o3352S(EDq-+jwGR}|RUv&zNUs*Iy3US`*n!$L7V9pftOAn?*kdtT&nboa$5|*{JB^h_1dUMf5*<GV{xD7p0(<Mn6wFZ>eKrZKDt?gq7aNpmypeUR-`2l{ppA?ksWfdxs9$=y+R}No!6B?DOA?iPvB>)%nXMQcOb=r@t~N?0)cO_{hE4;1y83x|bMfwCc<w<rbHyl!qhVhZR8hH6)?XyqTA|7xWe+GbNg<YgZsEzW_E&T+7ZMeOzn1DX@#-&agQf<g*6tcNTdUBw0ZIYuDp(*AC#sToLZyE7iwDcsvCrmn$e5jxUcR9x{@IwMqG}CaM-xVl2{p9+HoyHNBkq6KW463Gsz~K{C2TJdu6@Av%GHFHDjli`fUE60C&%KIYrewJoNTd&9M;aUiT=3#aP852XYO^`T+F>zs7H{!CJW!rY@zCMbe5dXw)QBNqKbeV`;&4I^>+uB4*Ix*dRV;VZ}zDnk#iC~?tqNRiX<sC+6J4@O%x@^Uv-Nd0M@r2Q&=&l-_)4qv48T4>P>-+D9Laun86m?9-=ZSiI11pcfsvX7au<k&&j?8&1{9d*?`o)CFK2YoWG+=zUO|>#=~F`;-gJPi;(jrf16Q5xcC^H58qv$-3`y{0qpJguUy(1eUS|w@9TTxKr#J7PBBliiDMtybpRo66&C7Q>IXQSTu_WP30YWF8E@BZG6QGt-`@=1V+X-I*mAc&s-qLp3mf~h#n+FwofS~iM}{%S|J8Du_ePCW5gR2Yf}2lQ11deSw*3vATOd}|#h2#SoxHPRz9Qe{BfZf#2T<hGX%OmW$L{rrOmGK_j4M4xHZV=UFGN?vF>ESQGR`C{hPqfzA8Hq%nP$Eos(;Ajc;E6mc)n*ik-VRKd=Q#aItquOM_YyrFgvA-e&Z<%O1C**alW{|zXqgt+}9;dp{E{%c_>|74&9`0T3s=HY)(2P2mmVxSY!rvU){&S{{i$F(k=')).decode('utf-8')
(WORK / 'edge_joint_engine.py').write_text(addon_source, encoding='utf8')
print('Embedded engines written:', len(base_source), len(addon_source))

In [ ]:
RESULT_DIR = WORK / 'EdgeIIoT_Joint_Results'
RESULT_DIR.mkdir(parents=True, exist_ok=True)

def run_stage(args, name):
    logfile = RESULT_DIR / f'{name}.log'
    with logfile.open('w', encoding='utf8') as handle:
        completed = subprocess.run([sys.executable, '-u', str(WORK / 'edge_joint_engine.py'), *args],
                                   cwd=str(WORK), env=ENV, stdout=handle, stderr=subprocess.STDOUT)
    output = logfile.read_text(errors='replace')
    print(f'{name}: exit code {completed.returncode}')
    print(output[-6500:], flush=True)
    if completed.returncode != 0:
        raise RuntimeError(f'{name} failed, exit={completed.returncode}; inspect {logfile}')

run_stage(['--mode','prepare'], 'prepare')

In [ ]:
# CPU histogram aggregation and CPU LightGBM: two parallel workers, T4 GPUs are not required.
from concurrent.futures import ThreadPoolExecutor
WORKERS = 2

def launch_worker(i):
    log = RESULT_DIR / f'worker_{i}.log'
    with log.open('w', encoding='utf8') as handle:
        proc = subprocess.run([sys.executable,'-u',str(WORK / 'edge_joint_engine.py'),
                               '--mode','worker','--worker',str(i),'--workers',str(WORKERS)],
                              env=ENV,cwd=str(WORK),stdout=handle,stderr=subprocess.STDOUT)
    return i,proc.returncode,log

with ThreadPoolExecutor(max_workers=WORKERS) as pool:
    worker_results = list(pool.map(launch_worker, range(WORKERS)))
for idx,code,log in worker_results:
    print('Worker:',idx,'Exit code:',code,'Log tail:')
    print(log.read_text(errors='replace')[-3500:])
if any(code for _,code,_ in worker_results):
    raise RuntimeError('One or more workers failed. Inspect worker logs and worker_*_failures.json')

In [ ]:
run_stage(['--mode','aggregate','--workers',str(WORKERS)],'aggregate')
archive = WORK / 'EdgeIIoT_Joint_Results.zip'
assert archive.exists(), 'Results ZIP missing'
print('RESULTS ZIP:',archive,'bytes:',archive.stat().st_size)
print('RUN STATUS:',(RESULT_DIR/'run_status.json').read_text())
print('EXTERNAL STATUS:',(RESULT_DIR/'external_domain_status.json').read_text())
from IPython.display import FileLink, display
display(FileLink(str(archive)))

## Interpretation protocol

Compare **FedHistogramGB vs LocalLightGBM** to test collaborative statistics-sharing benefits. Use **PooledLightGBM** as a centralized *upper-bound-style reference*, not a federated baseline or guaranteed upper bound. Track actual test FPR separately from 1%/5% source-validation target FPR.

Inspect `summary_mean_std.csv`, `all_metrics.csv`, `paired_method_deltas.csv`, `all_resources.csv`, and `fixed_test_manifest.csv`. `external_domain_status.json` says whether external evaluation ran; `external_transfer_results.csv` is created only with an independent feature-compatible dataset. Feature-name compatibility alone does not establish equivalent measurement semantics.

The experiment uses **federated decision stumps**; it does not implement privacy-preserving cryptographic protocols or federated full-depth boosted trees. Performance differences also reflect model-capacity differences.